# Project 01 — Data cleaning

## First-Time Buyer Affordability Pressure by Area

This notebook turns the ONS affordability workbook into an analysis-ready area-year dataset.

The first profiling stage confirmed that the workbook contains multiple tables across separate sheets. For the first-time buyer angle, the cleaning process focuses on local-authority lower-quartile measures: lower-quartile house price, lower-quartile workplace-based earnings and the lower-quartile affordability ratio.


## Cleaning approach

The workbook uses one sheet per measure and stores years across columns. This notebook reshapes the selected sheets into a long area-year format, merges the measures, applies basic data quality checks and saves cleaned outputs for SQL, Python, Excel and Power BI.


In [ ]:
from pathlib import Path
from datetime import date
import re

import numpy as np
import pandas as pd


In [ ]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
DICTIONARY_DIR = PROJECT_ROOT / "data" / "dictionary"
REPORTS_DIR = PROJECT_ROOT / "reports"

RAW_FILE = RAW_DIR / "aff1ratioofhousepricetoworkplacebasedearnings.xlsx"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
DICTIONARY_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Raw workbook not found: {RAW_FILE}. Run notebook 01 first."
    )

print(f"Using raw workbook: {RAW_FILE.relative_to(PROJECT_ROOT)}")


## Selected sheets

The workbook contents show that the local-authority lower-quartile tables are held on sheets 6a, 6b and 6c. The median affordability ratio is also loaded as a comparison measure.


In [ ]:
SHEET_CONFIG = {
    "6a": "lower_quartile_house_price",
    "6b": "lower_quartile_annual_earnings",
    "6c": "lower_quartile_affordability_ratio",
    "5c": "median_affordability_ratio",
}

ID_COLUMNS = [
    "country_region_code",
    "country_region_name",
    "local_authority_code",
    "local_authority_name",
]


In [ ]:
def normalise_column_name(column):
    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column).strip("_")
    return column


def extract_year_columns(columns):
    year_columns = []
    for column in columns:
        column_text = str(column).strip()
        if re.fullmatch(r"\d{4}(\.0)?", column_text):
            year_columns.append(column)
    return year_columns


def read_measure_sheet(sheet_name, measure_name):
    raw = pd.read_excel(RAW_FILE, sheet_name=sheet_name, header=1)
    raw = raw.dropna(how="all").dropna(axis=1, how="all")

    raw_columns = list(raw.columns)
    rename_map = {
        raw_columns[0]: "country_region_code",
        raw_columns[1]: "country_region_name",
        raw_columns[2]: "local_authority_code",
        raw_columns[3]: "local_authority_name",
    }
    raw = raw.rename(columns=rename_map)

    year_columns = extract_year_columns(raw.columns)

    long = raw[ID_COLUMNS + year_columns].melt(
        id_vars=ID_COLUMNS,
        value_vars=year_columns,
        var_name="year",
        value_name=measure_name,
    )

    long["year"] = long["year"].astype(str).str.replace(".0", "", regex=False).astype(int)
    long[measure_name] = (
        long[measure_name]
        .replace({"[x]": np.nan, "x": np.nan, ":": np.nan})
        .pipe(pd.to_numeric, errors="coerce")
    )

    long["source_sheet"] = sheet_name
    return long


In [ ]:
measure_frames = []

for sheet_name, measure_name in SHEET_CONFIG.items():
    measure_frame = read_measure_sheet(sheet_name, measure_name)
    print(f"Loaded sheet {sheet_name}: {measure_name} — {measure_frame.shape[0]:,} rows")
    measure_frames.append(measure_frame.drop(columns="source_sheet"))


## Merge measures into one area-year table


In [ ]:
cleaned = measure_frames[0]

for frame in measure_frames[1:]:
    cleaned = cleaned.merge(
        frame,
        on=ID_COLUMNS + ["year"],
        how="outer",
        validate="one_to_one",
    )

cleaned = cleaned.sort_values(["local_authority_name", "year"]).reset_index(drop=True)

print(f"Cleaned rows: {cleaned.shape[0]:,}")
print(f"Cleaned columns: {cleaned.shape[1]:,}")
cleaned.head()


## Data quality checks


In [ ]:
duplicate_key_count = cleaned.duplicated(subset=["local_authority_code", "year"]).sum()
missing_key_counts = cleaned[["local_authority_code", "local_authority_name", "year"]].isna().sum()

quality_summary = pd.DataFrame([
    {"check": "row_count", "result": cleaned.shape[0]},
    {"check": "column_count", "result": cleaned.shape[1]},
    {"check": "duplicate_local_authority_year_keys", "result": duplicate_key_count},
    {"check": "missing_local_authority_code", "result": int(missing_key_counts["local_authority_code"])},
    {"check": "missing_local_authority_name", "result": int(missing_key_counts["local_authority_name"])},
    {"check": "missing_year", "result": int(missing_key_counts["year"])},
])

quality_summary


In [ ]:
missing_profile = (
    cleaned
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
    .rename(columns={"index": "column_name", 0: "missing_pct"})
)

missing_profile


## Add analytical fields


In [ ]:
def assign_pressure_band(ratio):
    if pd.isna(ratio):
        return np.nan
    if ratio < 5:
        return "Lower pressure"
    if ratio < 8:
        return "Moderate pressure"
    if ratio < 12:
        return "High pressure"
    return "Severe pressure"


cleaned["affordability_pressure_band"] = cleaned["lower_quartile_affordability_ratio"].apply(assign_pressure_band)

latest_year = int(cleaned["year"].max())
latest = cleaned.loc[cleaned["year"] == latest_year].copy()
latest["latest_year_rank_most_pressured"] = latest["lower_quartile_affordability_ratio"].rank(
    method="dense",
    ascending=False,
)

cleaned = cleaned.merge(
    latest[["local_authority_code", "latest_year_rank_most_pressured"]],
    on="local_authority_code",
    how="left",
)

print(f"Latest year in dataset: {latest_year}")
cleaned.head()


## Quick review of latest-year affordability pressure


In [ ]:
latest_review_columns = [
    "country_region_name",
    "local_authority_name",
    "lower_quartile_house_price",
    "lower_quartile_annual_earnings",
    "lower_quartile_affordability_ratio",
    "median_affordability_ratio",
    "affordability_pressure_band",
]

latest_most_pressured = (
    cleaned.loc[cleaned["year"] == latest_year, latest_review_columns]
    .dropna(subset=["lower_quartile_affordability_ratio"])
    .sort_values("lower_quartile_affordability_ratio", ascending=False)
    .head(10)
)

latest_most_pressured


In [ ]:
latest_least_pressured = (
    cleaned.loc[cleaned["year"] == latest_year, latest_review_columns]
    .dropna(subset=["lower_quartile_affordability_ratio"])
    .sort_values("lower_quartile_affordability_ratio", ascending=True)
    .head(10)
)

latest_least_pressured


## Save cleaned outputs


In [ ]:
cleaned_output = CLEANED_DIR / "affordability_area_year_cleaned.csv"
quality_output = CLEANED_DIR / "cleaning_quality_summary.csv"
missing_output = CLEANED_DIR / "cleaned_missing_profile.csv"

cleaned.to_csv(cleaned_output, index=False)
quality_summary.to_csv(quality_output, index=False)
missing_profile.to_csv(missing_output, index=False)

print(f"Saved cleaned data to: {cleaned_output.relative_to(PROJECT_ROOT)}")
print(f"Saved quality summary to: {quality_output.relative_to(PROJECT_ROOT)}")
print(f"Saved missing profile to: {missing_output.relative_to(PROJECT_ROOT)}")


In [ ]:
data_dictionary = pd.DataFrame([
    {"column_name": "country_region_code", "description": "ONS code for the country or English region", "type": "string"},
    {"column_name": "country_region_name", "description": "Country or English region name", "type": "string"},
    {"column_name": "local_authority_code", "description": "ONS local authority code", "type": "string"},
    {"column_name": "local_authority_name", "description": "Local authority name", "type": "string"},
    {"column_name": "year", "description": "Reference year", "type": "integer"},
    {"column_name": "lower_quartile_house_price", "description": "Lower-quartile house price from ONS affordability workbook", "type": "numeric"},
    {"column_name": "lower_quartile_annual_earnings", "description": "Lower-quartile gross annual workplace-based earnings", "type": "numeric"},
    {"column_name": "lower_quartile_affordability_ratio", "description": "Lower-quartile house price divided by lower-quartile annual earnings", "type": "numeric"},
    {"column_name": "median_affordability_ratio", "description": "Median house price divided by median annual earnings, included as a comparison measure", "type": "numeric"},
    {"column_name": "affordability_pressure_band", "description": "Banding derived from the lower-quartile affordability ratio", "type": "string"},
    {"column_name": "latest_year_rank_most_pressured", "description": "Latest-year dense rank, with 1 assigned to the highest affordability pressure", "type": "numeric"},
])

dictionary_output = DICTIONARY_DIR / "data_dictionary.csv"
data_dictionary.to_csv(dictionary_output, index=False)

data_dictionary


## Cleaning summary

The cleaned dataset is ready for the SQL analysis stage once the row counts, missing profile and latest-year rankings have been reviewed.
